<h2>Graphs</h2>

The easiest way to represent graphs are via dictionary where keys are node and values are list of neighbors. Note, for graphs where nodes don't have any edges, you still need to put them in dict but without any neighbors.
If the graph needs to contain prev pointers, visited marks etc, then values can be yet another dict, or you can define parallel dicts.

<h3>Dijkstra's algo</h3>

Finding shortest path from one node to another.

In [58]:
# DIRECTED GRAPH:
# a -> b -> c -> d -> e
# a -> d
# ab = 2, bc = 3, cd = 4, de = 5, ad = 1
directed_graph = {
    "a": [("b", 2), ("d", 1)],
    "b": [("c", 3)],
    "c": [("d", 4)],
    "d": [("e", 5)],
    "e": [], # Must specify otherwise graph will not have 'e' node
}

# UNDIRECTED GRAPH:
# a - b - c - d - e
# a - d
# ab = 2, bc = 3, cd = 4, de = 5, ad = 1
undirected_graph = {
    "a": [("b", 2), ("d", 1)],
    "b": [("c", 3), ("a", 2)],
    "c": [("d", 4), ("b", 3)],
    "d": [("e", 5), ("c", 4), ("a", 1)],
    "e": [("d", 5)],
}

In [59]:
import heapq

In [60]:
def dijkstra(graph: dict[str, tuple[str, int]], start: str, end: str) -> list[str]:
    queue = []
    dists = {}
    prevs = {}

    if start not in graph or end not in graph:
        # start or end nodes are not in the graph
        return []

    heapq.heappush(queue, (0, start))
    prevs[start] = None
    dists[start] = 0

    while queue:
        cur_dist, next_node = heapq.heappop(queue)
        print(f"considering {next_node}")
        if next_node == end:
            # end reached, construct the path and return
            path = [end]
            while end in prevs:
                end = prevs[end]
                path.append(end)
                if prevs[end] is None:
                    break
            print(f"Found the shortest path with distance {cur_dist}")
            return path[::-1]

        if next_node in dists and dists[next_node] < cur_dist:
            # skip, shorter dist is already detected for next_node
            continue

        for entry in graph[next_node]:
            neighbor, dist = entry
            if neighbor not in dists or dists[neighbor] > cur_dist + dist:
                dists[neighbor] = cur_dist + dist
                prevs[neighbor] = next_node
                heapq.heappush(queue, (dists[neighbor], neighbor))

    return []

In [61]:
dijkstra(directed_graph, "a", "e")

considering a
considering d
considering b
considering c
considering e
Found the shortest path with distance 6


['a', 'd', 'e']

In [62]:
dijkstra(undirected_graph, "a", "e")

considering a
considering d
considering b
considering c
considering e
Found the shortest path with distance 6


['a', 'd', 'e']

Dijkstra's longest path can be achieved by exactly the same algorithm, just the distances have to be negated. The following code is the same as the above, with only 2 slight changes.

In [69]:
import time

In [80]:
# dijkstra for maximum path
def dijkstra_max(graph: dict[str, tuple[str, int]], start: str, end: str) -> list[str]:
    queue = []
    dists = {}
    prevs = {}

    if start not in graph or end not in graph:
        # start or end nodes are not in the graph
        return []

    heapq.heappush(queue, (0, start))
    prevs[start] = None
    dists[start] = 0

    while queue:
        cur_dist, next_node = heapq.heappop(queue)
        print(f"considering {next_node}")
        if next_node == end:
            # end reached, construct the path and return
            path = [end]
            while end in prevs:
                end = prevs[end]
                path.append(end)
                if prevs[end] is None:
                    break
            # NEGATE WHEN REPORTING - CHANGE 1
            print(f"Found the longest path with distance {-cur_dist}")
            return path[::-1]

        if next_node in dists and dists[next_node] < cur_dist:
            # skip, shorter dist is already detected for next_node
            continue

        for entry in graph[next_node]:
            neighbor, dist = entry
            dist = (
                -dist
            )  # CONSIDER NEGATIVE DISTS  - CHANGE 2 -- all else is the same as above
            if neighbor not in dists or dists[neighbor] > cur_dist + dist:
                dists[neighbor] = cur_dist + dist
                prevs[neighbor] = next_node
                heapq.heappush(queue, (dists[neighbor], neighbor))

    return []

In [81]:
# This will work because directed graph is a DAG
dijkstra_max(directed_graph, "a", "e")

considering a
considering b
considering c
considering d
considering e
Found the longest path with distance 14


['a', 'b', 'c', 'd', 'e']

In [82]:
# This will run forever because undirected graph is not a DAG and we end up in an infinite loop
dijkstra_max(undirected_graph, "a", "e")

considering a
considering b
considering c
considering d
considering e


KeyboardInterrupt: 

<h2>Prim's Algorithm</h2>

Prim's algorithm is to find minimum spanning tree, i.e. a DAG with minumum sum of edges that connect all nodes. It is implemented by repeatedly choosing nodes that are outside of current DAG and which have minimum distance from it. First node can be chosen randomly.
We will use priority queue to store edges of the graph so that the minimum edge can be retrieved.

In [93]:
def prim(graph: dict[str, tuple[str, int]], is_directed=False) -> dict[str, tuple[str, int]]:
    dag = {}
    queue = []
    
    # Picking the first node as a start 
    for start_node in graph.keys():
        break
    
    # Pushing first node's edges to our PQueue
    for pair in graph[start_node]:
        neighbor, dist = pair
        heapq.heappush(queue, (dist, start_node, neighbor))
    
    dag[start_node] = []
    # Repeatedly picking minimum edges until queue is empty or dag contains all nodes (in which case MST is constructed)
    while queue or len(dag) < len(graph):
        dist, from_node, to_node = heapq.heappop(queue)
        if from_node in dag and to_node in dag:
            # both nodes are already in MST
            continue
        if from_node in dag:
            # Adding to_node to the DAG, from_node is already in the DAG
            if is_directed:
                dag[to_node] = []
            else:
                dag[to_node] = [(from_node, dist)]
            dag[from_node].append((to_node, dist))
            for pair in graph[to_node]:
                neighbor, d = pair
                heapq.heappush(queue, (d, to_node, neighbor))
        elif not is_directed:
            # Adding from_node to DAG, to_node is already in the DAG
            dag[from_node] = [(to_node, dist)]
            dag[to_node].append((from_node, dist))
            for pair in graph[from_node]:
                neighbor, d = pair
                heapq.heappush(queue, (d, from_node, neighbor))
    
    if len(dag) == len(graph):
        return dag
    else:
        print(f"No minimum spanning tree exists, so far could identify {dag}")
        return None
    

In [94]:
prim(undirected_graph)

{'a': [('d', 1), ('b', 2)],
 'd': [('a', 1), ('e', 5)],
 'b': [('a', 2), ('c', 3)],
 'c': [('b', 3)],
 'e': [('d', 5)]}

In [96]:
prim(directed_graph, is_directed=True)

{'a': [('d', 1), ('b', 2)], 'd': [('e', 5)], 'b': [('c', 3)], 'c': [], 'e': []}